# A Reproducible Small-Scale Study: From Ethics Review to Evidence Chain

The preceding tutorials each cover one method—difference-in-differences, complex sampling, text analysis. This notebook chains their underlying machinery into a single **complete yet minimal policy evaluation**: a policy rolled out to a set of firms in 2015, and we aim to estimate its **average treatment effect on the treated (ATT)** on firm outcome variables, while ensuring the entire analysis can **account for itself** after completion: where the data come from, whether it passed ethics review, what identification assumptions were used, and whether the conclusion is licensed to be read causally.

This approach is necessary because credibility in social science rests not on a point estimate per se, but on the entire apparatus surrounding it. Anyone can compute `-0.73`; what actually determines whether it makes it into a paper is whether you **confirmed you could access this data**, whether you **tested the parallel trends assumption**, whether the conclusion **holds under different standard error specifications**, and whether **others can reproduce your analysis from your records**. This notebook's focus is therefore not "how to estimate DID"—that is covered in [02_causal_did](02_causal_did.ipynb)—but rather **how to document an analysis as a reproducible working paper**.

The entire workflow is completed using `socialverse`. It is an analysis library for social science: each method is a registered function that validates at runtime whether its prerequisites are satisfied, and silently logs each step into a lineage ledger. The data are built-in synthetic panel (40 firms × 8 years, true ATT = −0.8), clean and controllable, letting us examine each step clearly. The sequence we follow is: **pass ethics review → ingest data → declare design → test parallel trends → estimate ATT → visualize → export evidence chain**.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件

import json
import os

import pandas as pd

import socialverse as sv
from socialverse import datasets as ds

# 图存到 notebook 同目录,markdown 里的 ![](fig_xxx.png) 才能就近解析,
# 无论从哪个目录跑脚本都一致
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # 交互式内核里没有 __file__
    _HERE = os.getcwd()


def figpath(name: str) -> str:
    return os.path.join(_HERE, name)

## Declare Study Objectives

A reproducible study does not begin with loading data and running code; rather, it starts by clearly stating what you wish to estimate. We seek to estimate the **average treatment effect on the treated (ATT)** (not mere correlation), with outcome variable `y`. Both derive from the research question itself—no function can generate them for you—so we record them from the start in the study state `StudyState`, which serves as a shared notebook for this analysis. Each subsequent step reads context from this state and writes results back to it.

In [2]:
st = sv.StudyState()
st.write("estimand", "target", "ATT")   # 目标量:平均处理效应,不是相关
st.write("variables", "outcome", "y")   # 结果变量列名

print("研究状态初始化:", repr(st))

研究状态初始化: StudyState(variables[1], estimand[1]; 0 steps)


## Ingest Panel Data

The data are a **firm × year** long-format panel: each row is an observation of one firm in one year. Firms in the treatment group are hit by the policy in the year marked by `first_treated` (2015); `treat_post` marks observations that have received treatment; `y` is the outcome variable. The data were generated with an injected true ATT = −0.8, and we will see whether our estimate recovers it.

After ingestion, we register the table in the study state using `sv.pp.ingest`, so all subsequent steps read data from the state rather than passing it manually.

In [3]:
df = ds.load_did_panel(att=-0.8)
sv.pp.ingest(st, data=df, name="did_panel_firm_year")

print("面板维度:", df.shape, "| 列:", list(df.columns))
df.head(6)

面板维度: (320, 8) | 列: ['firm_id', 'year', 'treat', 'post', 'treat_post', 'first_treated', 'y', 'x1']


,firm_id,year,treat,post,treat_post,first_treated,y,x1
0,0,2010,1,0,0,2015,1.6221,1.5139
1,0,2011,1,0,0,2015,2.8355,0.7813
2,0,2012,1,0,0,2015,2.5744,-0.3139
3,0,2013,1,0,0,2015,3.2232,1.9603
4,0,2014,1,0,0,2015,3.5321,1.3151
5,0,2015,1,1,1,2015,2.3258,-1.2083


## Governance Gate: Pass Ethics Review First

In social science research, **whether you can access this data** is prior to the question of how to analyze it. `ethics_check` performs genuine checks on four things—IRB classification, informed consent, **identifiability (computing k-anonymity)**, and data minimization—returning a single judgment: `PASS / FIX / NO-GO`. It requires you first to declare what your **unit of analysis** is, because what "one row of data represents" directly determines whether the analysis touches personal data.

We will deliberately commit a common mistake: record the unit as `"firm-year"` (with a hyphen) and see how the gate responds.

In [4]:
st.write("design", "unit", "firm-year")   # 手滑:连字符写法
sv.gov.ethics_check(st, data=df, quasi_identifiers=["treat"])

ethics = st.governance["ethics"]
print("伦理判决:", ethics["verdict"], "| 判定为人类受试者:", ethics["human_subjects"])
for c in ethics["checks"]:
    print(f"  - {c['check']:<12} {c['status']:<6} :: {c['detail']}")

伦理判决: NO-GO | 判定为人类受试者: True
  - irb          NO-GO  :: human subjects but no IRB determination recorded
  - consent      NO-GO  :: human subjects but no consent basis recorded
  - k_anonymity  PASS   :: k=160 ≥ threshold 5
  - minimization FIX    :: confirm direct identifiers dropped and variables minimized to need


The judgment is **NO-GO**, and it classified the unit as "human subjects." The reason is straightforward: `"firm-year"` is not in its recognized set of non-human units (`firm`, `country`, `document` are, but hyphenated `firm-year` is not), so it conservatively treats it as potentially involving personal data, issuing NO-GO in the absence of IRB documentation and informed consent. This is not a bug but a **conservative default**: when in doubt, apply the strictest standard. The actual blockers are IRB and informed consent—the k-anonymity check itself passes.

The correct fix is to truthfully declare the unit as `"firm"` and supply governance facts: this is a firm panel, sourced from public regulatory data, already minimized to required variables. We rerun the gate.

In [5]:
st.write("design", "unit", "firm")        # 如实声明:分析单元是「企业」
sv.gov.ethics_check(
    st,
    data=df,
    quasi_identifiers=["treat", "year"],  # 用组合准标识做真 k-匿名
    irb="exempt",                          # 企业监管面板:IRB 豁免
    consent="public",                      # 公开数据,无需个体知情同意
    minimized=True,                        # 已删除直接标识,只留所需变量
)
ethics = st.governance["ethics"]
print("修正后判决:", ethics["verdict"], "| 人类受试者:", ethics["human_subjects"])
for c in ethics["checks"]:
    print(f"  - {c['check']:<12} {c['status']}")
print("k-匿名 k =", ethics["k_anonymity"]["k"],
      "(阈值", ethics["k_anonymity"]["k_threshold"], ")")

assert ethics["verdict"] == "PASS", "闸门未放行,不应继续分析"
print("闸门放行 ✓ —— 现在才被允许进入因果分析")

修正后判决: PASS | 人类受试者: False
  - irb          PASS
  - consent      PASS
  - k_anonymity  PASS
  - minimization PASS
k-匿名 k = 20 (阈值 5 )
闸门放行 ✓ —— 现在才被允许进入因果分析


The judgment is now **PASS**; the analysis unit is correctly identified as non-human (firms), all four checks are green, and k-anonymity k = 20 exceeds the threshold. The gate is part of the chain itself, not a compliance document tacked on afterward—this will be evident when we export the evidence chain at the end.

## Declare Study Design

Having passed the gate, the next step is to translate **concrete column names** from the data into roles that causal analysis can read: which is the unit, which is time, which column marks treatment, which records the first year of treatment. `declare_design` registers this mapping in the study state, so subsequent estimators read these roles rather than guessing column names. It also validates that these columns actually exist in the data.

In [6]:
sv.pp.declare_design(
    st,
    panel_id="firm_id",
    time="year",
    treatment="treat_post",       # 处理×时点交互(数据里已预先算好)
    first_treated="first_treated",
)
for k in ("unit", "panel_id", "time", "treatment", "first_treated"):
    print(f"  design['{k}'] = {st.design.get(k)!r}")
print("列名校验警告:", st.design.get("warnings", "无(所有列都在数据中)"))

  design['unit'] = 'firm'
  design['panel_id'] = 'firm_id'
  design['time'] = 'year'
  design['treatment'] = 'treat_post'
  design['first_treated'] = 'first_treated'
列名校验警告: 无(所有列都在数据中)


## Test Parallel Trends

This is the critical threshold of the causal chain. DID can be read causally only if the **parallel trends** assumption holds: absent the policy, the treatment and control groups would have evolved along parallel trajectories. This premise is unobservable directly, but we can examine it indirectly using "pre-trends" from pre-treatment periods—`parallel_trends` estimates an event study with unit and time fixed effects, then performs a joint Wald test on all pre-treatment relative-period coefficients.

The null hypothesis is "all pre-treatment coefficients equal zero," i.e., parallel pre-trends across groups. If `p > 0.05` (fail to reject), we judge `pass` and the identification assumption holds; if `p` is small, pre-trends already diverged, and even if we obtain a coefficient, it should not be called "causal." Our synthetic data were generated with genuine parallel pre-trends and should pass.

In [7]:
sv.tl.parallel_trends(st)

pt = st.diagnostics["pretrend"]
print(f"联合 F = {pt['joint_F']:.3f}   p = {pt['p_value']:.3f}   (前导期数 = {pt['n_pre']})")
print("判定:", st.identification["parallel_trends"], "—", pt["note"])
print("\n前导期系数(应都≈0):")
for period, (coef, se) in pt["pre_coefs"].items():
    print(f"  event_time={period:>3}:  β={coef:+.4f}  (se={se:.4f})")

联合 F = 0.474   p = 0.755   (前导期数 = 4)
判定: pass — 未拒绝平行趋势(p>0.05)

前导期系数(应都≈0):
  event_time= -5:  β=+0.0236  (se=0.1764)
  event_time= -4:  β=-0.1120  (se=0.1653)
  event_time= -3:  β=-0.2264  (se=0.1884)
  event_time= -2:  β=-0.1447  (se=0.2262)


`p = 0.755` is far above 0.05—the joint test on pre-treatment coefficients is insignificant, and parallel trends hold. Lead-period coefficients cluster near zero. This determination is recorded in the study state and becomes a prerequisite for whether `did` will run: without it, `did` refuses to execute.

## Estimate ATT

Now we can estimate. `did` fits a two-way fixed effects model (`y ~ treat_post + unit fixed effects + time fixed effects`), computing robust standard errors clustered at `firm_id`—inference on treatment effects typically requires clustering at the unit level to allow for within-firm serial correlation. It simultaneously reads the parallel trends determination into its conclusion: if passed, it labels the result "causal ATT"; if not, it downgrades to "association, not causal."

In [8]:
sv.tl.did(st)

m = st.models["did"]
print(f"ATT    = {m['att']:+.4f}   (真值注入 = -0.8000)")
print(f"SE     = {m['se']:.4f}   (聚类于 {m['n_clusters']} 家企业)")
print(f"95% CI = [{m['ci'][0]:.4f}, {m['ci'][1]:.4f}]")
print(f"p      = {m['p']:.2e}")
print(f"n      = {m['n']}   估计量 = {m['estimator']}")
print("结论   :", m["note"])

ATT    = -0.7307   (真值注入 = -0.8000)
SE     = 0.1021   (聚类于 40 家企业)
95% CI = [-0.9308, -0.5307]
p      = 8.15e-13
n      = 320   估计量 = twfe_ols_cluster
结论   : 因果 ATT(平行趋势通过)


The estimated ATT ≈ **−0.73** with 95% confidence interval `[−0.93, −0.53]` lying entirely in the negative range, not crossing 0, and encompassing the true injected value −0.8. Because parallel trends passed, this number is **licensed** to be read as a causal ATT: the policy caused a significant decline in the outcome variable.

## Robustness: Sensitivity of Standard Errors to Specification

With the point estimate in place, a routine check is whether standard errors are stable across different variance specifications. `did` re-estimates the same ATT under three specifications—classical (homoskedastic), heteroskedasticity-robust (HC1), and firm-clustered. The point estimate is identical across all three (variance specification does not affect the point estimate); the clustered SE is typically the largest and most credible because it allows for within-firm serial correlation.

In [9]:
rob = pd.DataFrame(st.diagnostics["robustness"]["specs"])
rob

,spec,att,se,p
0,classical,-0.730739,0.094007,1.581385e-13
1,HC1_robust,-0.730739,0.092468,2.730834e-15
2,cluster_panel,-0.730739,0.102079,8.154308e-13


## Unpack Dynamic Effects

ATT is an average. Event study decomposes it by relative treatment timing: using the period immediately before treatment (−1) as the reference, we obtain coefficients for each relative period. Pre-treatment coefficients let us re-examine whether pre-trends adhere to zero; post-treatment coefficients characterize how the effect evolves over time after policy implementation.

In [10]:
sv.tl.event_study(st)

es = st.models["event_study"]
pd.DataFrame(
    [{"相对时点": int(k), "系数": round(v[0], 3), "SE": round(v[1], 3)}
     for k, v in es["coefs"].items()]
)

,相对时点,系数,SE
0,-5,0.024,0.176
1,-4,-0.112,0.165
2,-3,-0.226,0.188
3,-2,-0.145,0.226
4,-1,0.000,0.000
5,0,-0.888,0.154
6,1,-0.786,0.162
7,2,-0.794,0.175


## Visualization

Once results are ready, plotting functions read numbers directly from the study state and generate figures without manual assembly. We produce two plots: a forest plot of the ATT and an event-study plot of the dynamic effect. Both are saved as PNG files in the same directory.

In [11]:
sv.pl.forest(st, out=figpath("fig_study.png"),
             title="政策 ATT · 点估计 ± 95% CI")
sv.pl.event_study_plot(st, out=figpath("fig_eventstudy.png"))
print("图已保存:fig_study.png, fig_eventstudy.png")

图已保存:fig_study.png, fig_eventstudy.png


**Forest plot**—single ATT coefficient with 95% confidence interval, dashed line indicates zero effect reference:

![Forest plot](fig_study.png)

**Event-study plot**—lead periods cluster at zero (parallel trends hold), jumping to approximately −0.8 at treatment time 0 and persisting:

![Event-study plot](fig_eventstudy.png)

## Export Evidence Chain

At this point the study is "complete," but the true deliverable of a reproducible study is not just the −0.73, but rather **a chain that proves its own provenance**. By now, the study state has automatically accumulated an **append-only lineage ledger**: each time a function executes successfully, it appends a record—the step number, which slots it consumed, which it produced. You wrote no logging code; the ledger is a byproduct of the analysis.

Printing it reveals a chain from "ethics clearance" through "ATT estimation," with dependencies locked together end to end.

In [12]:
prov_rows = []
for p in st.provenance:
    prov_rows.append({
        "step": p["step"],
        "function": p["function"].split(".")[-1],
        "requires": ", ".join(f"{s}.{k}" for s, ks in p["requires"].items() for k in ks) or "∅",
        "produces": ", ".join(f"{s}.{k}" for s, ks in p["produces"].items() for k in ks) or "∅",
    })
pd.DataFrame(prov_rows)

,step,function,requires,produces
0,1,ingest,∅,sources.datasets
1,2,ethics_check,design.unit,governance.ethics
2,3,ethics_check,design.unit,governance.ethics
3,4,declare_design,sources.datasets,"design.panel_id, design.time, design.treatment..."
4,5,parallel_trends,"design.panel_id, design.time, design.treatment...","diagnostics.pretrend, identification.parallel_..."
5,6,did,"design.panel_id, design.time, design.treatment...","models.did, models.twfe, diagnostics.robustness"
6,7,event_study,"design.panel_id, design.time, design.treatment...",models.event_study
7,8,forest,models.did,artifacts.figures
8,9,event_study_plot,models.event_study,artifacts.figures


Reading the chain: each step's `produces` exactly feeds the next step's `requires`—`ingest` produces `sources.datasets` for `declare_design` to use; `declare_design` produces `design.*` for `parallel_trends`; `parallel_trends` produces `identification.parallel_trends`, which is exactly what `did` needs. Dependencies lock together end to end with no gaps—this is the machine-readable form of "reproducibility." Notice `ethics_check` appears **twice** (the initial erroneous declaration plus the corrected version)—the ledger is append-only, faithfully recording even "made a mistake then fixed it"; moreover, ethics review precedes causal analysis, proving governance was not tacked on afterward.

This ledger can be exported directly as JSON and archived in the paper's appendix as "analysis provenance."

In [13]:
prov_json_path = figpath("evidence_chain_provenance.json")
with open(prov_json_path, "w", encoding="utf-8") as fh:
    json.dump(st.provenance, fh, ensure_ascii=False, indent=2)
print("出处台账已导出:", os.path.basename(prov_json_path),
      f"({len(st.provenance)} 步)")

出处台账已导出: evidence_chain_provenance.json (9 步)


## Final Snapshot

`st.summary()` displays the entire research working paper at a glance: which slots are filled, what they contain, and how many steps produced them. This is what a reproducible study should leave behind when complete—not an isolated number, but a closed loop from **data sourcing → governance clearance → design declaration → identification assumption testing → estimation → visualization**, along with an archivable lineage ledger.

In [14]:
print(st.summary())

print("\n本次研究导出的可复核文件:")
for pth in (figpath("fig_study.png"), figpath("fig_eventstudy.png"), prov_json_path):
    exists = "✓" if os.path.exists(pth) else "✗"
    print(f"  [{exists}] {os.path.basename(pth)}")

StudyState
  sources: datasets, dataset_name, dataset_meta
  design: unit, panel_id, time, treatment, first_treated
  variables: outcome
  estimand: target
  identification: parallel_trends
  models: twfe, did, event_study
  diagnostics: pretrend, robustness
  governance: ethics
  artifacts: figures
  provenance: 9 step(s)

本次研究导出的可复核文件:
  [✓] fig_study.png
  [✓] fig_eventstudy.png
  [✓] evidence_chain_provenance.json


## Summary

We have completed a full policy evaluation loop: pass ethics gate → ingest → declare design → test parallel trends → estimate ATT → visualize → export evidence chain. Examining each step, each has a mature real-world counterpart: the governance gate ≈ IRB form plus k-anonymity from `sdcMicro`; causal estimation ≈ `pyfixest` or R's `fixest` and `did` (Callaway–Sant'Anna); visualization ≈ `matplotlib` or `coefplot`.

Compared to assembling these tools separately, `socialverse` adds two things, neither requiring extra code: parallel trends is a gate that **actually blocks you** (without passage, `did` refuses to run rather than silently handing you an unidentified coefficient), and a **lineage chain running throughout, append-only** (directly exportable as paper-appendix-grade analysis provenance). Other tools let you **produce** a result; `socialverse` makes the result carry its own provenance, dependencies, and governance record.

This concludes the notebook series. To revisit details of individual methods, you can return to the causal chapter [02_causal_did](02_causal_did.ipynb).